#Extraer el texto de Tik Tok

##1 — Instalar dependencias

In [1]:
TIKTOK_URL = "https://vt.tiktok.com/ZSataVEaX/"
#"https://vt.tiktok.com/ZSatjpvbJ/"
OUT_DIR = "/content/tiktok"

In [3]:
!pip -q install -U yt-dlp requests

##2 — Extraer metadata del video (yt-dlp JSON)

In [4]:
import os, json
import yt_dlp

In [5]:
os.makedirs(OUT_DIR, exist_ok=True)

ydl_opts = {
    "quiet": True,
    "no_warnings": True,
    "noplaylist": True,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(TIKTOK_URL, download=False)

raw_path = f"{OUT_DIR}/video_info_raw.json"
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print("RAW guardado:", raw_path)
print("Keys ejemplo:", list(info.keys())[:30])

RAW guardado: /content/tiktok/video_info_raw.json
Keys ejemplo: ['id', 'formats', 'subtitles', 'http_headers', 'channel', 'channel_id', 'uploader', 'uploader_id', 'channel_url', 'uploader_url', 'track', 'artists', 'duration', 'title', 'description', 'timestamp', 'view_count', 'like_count', 'repost_count', 'comment_count', 'save_count', 'thumbnails', 'original_url', 'webpage_url', 'webpage_url_basename', 'webpage_url_domain', 'extractor', 'extractor_key', 'playlist', 'playlist_index']


##3 — Parsear: música, track, artista, cuenta

In [6]:
def pick(*vals):
    for v in vals:
        if v not in (None, "", "NA"):
            return v
    return None

uploader = pick(info.get("uploader"), info.get("channel"), info.get("creator"))
uploader_id = pick(info.get("uploader_id"), info.get("channel_id"))
uploader_url = pick(info.get("uploader_url"), info.get("channel_url"), info.get("webpage_url_basename"))

track = pick(info.get("track"), info.get("alt_title"))
artist = pick(info.get("artist"), info.get("composer"))
# En muchos casos TikTok pone "original sound" sin artista; igual cuenta como audio/sonido.
has_music = bool(track or artist)

meta = {
    "tiktok_url": TIKTOK_URL,
    "id": info.get("id"),
    "title": info.get("title"),
    "duration_sec": info.get("duration"),
    "uploader_name": uploader,
    "uploader_id": uploader_id,
    "uploader_url": uploader_url,
    "has_music": has_music,
    "track": track,
    "artist": artist,
    "view_count": info.get("view_count"),
    "like_count": info.get("like_count"),
    "comment_count": info.get("comment_count"),
    "repost_count": info.get("repost_count"),
}

meta

{'tiktok_url': 'https://vt.tiktok.com/ZSataVEaX/',
 'id': '7582847487392632071',
 'title': '#perfume #fyp #poloblue #andrescroxatto #regalos  Polo Blue un perfum...',
 'duration_sec': 90,
 'uploader_name': 'capibara_store_oficial',
 'uploader_id': '7426875402553328646',
 'uploader_url': 'https://www.tiktok.com/@capibara_store_oficial',
 'has_music': True,
 'track': 'sonido original',
 'artist': 'Capibara Store',
 'view_count': 857500,
 'like_count': 29900,
 'comment_count': 318,
 'repost_count': 3696}

##4 — Intentar followers/seguidos desde el perfil (scraping “best effort”)

Esto puede fallar si TikTok bloquea la petición desde Colab. Si falla, igual dejamos todo lo demás OK.

In [7]:
import re, requests

In [8]:
def fetch_profile_stats(unique_id: str):
    if not unique_id:
        return {"followers": None, "following": None, "profile_name": None, "status": "no_uploader"}

    url = f"https://www.tiktok.com/@{unique_id}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
        "Referer": "https://www.tiktok.com/",
    }

    r = requests.get(url, headers=headers, timeout=20)
    if r.status_code != 200:
        return {"followers": None, "following": None, "profile_name": None, "status": f"http_{r.status_code}"}

    html = r.text

    # Busca followerCount/followingCount (aparecen en JSON embebido en la página)
    m_followers = re.search(r'"followerCount"\s*:\s*(\d+)', html)
    m_following = re.search(r'"followingCount"\s*:\s*(\d+)', html)
    m_nickname  = re.search(r'"nickname"\s*:\s*"([^"]+)"', html)

    return {
        "followers": int(m_followers.group(1)) if m_followers else None,
        "following": int(m_following.group(1)) if m_following else None,
        "profile_name": m_nickname.group(1) if m_nickname else None,
        "status": "ok" if (m_followers or m_following) else "parsed_none"
    }

# OJO: a veces uploader_id no es el @handle. Si uploader_url trae "@handle", lo extraemos.
unique_id_guess = None
if isinstance(uploader_url, str) and "@" in uploader_url:
    unique_id_guess = uploader_url.split("@")[-1].split("/")[0]
elif isinstance(uploader, str) and " " not in uploader and len(uploader) <= 40:
    unique_id_guess = uploader  # fallback

profile = fetch_profile_stats(unique_id_guess)
profile

{'followers': 13400,
 'following': 50,
 'profile_name': 'Capibara Store',
 'status': 'ok'}

##5 — Unir todo y guardar metadata.json

In [ ]:
import json

In [9]:
final = {**meta, **{
    "profile_unique_id_guess": unique_id_guess,
    "profile_name": profile.get("profile_name"),
    "followers": profile.get("followers"),
    "following": profile.get("following"),
    "profile_status": profile.get("status"),
}}

out_path = f"{OUT_DIR}/metadata.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=2)

print("✅ Guardado:", out_path)
final

✅ Guardado: /content/tiktok/metadata.json


{'tiktok_url': 'https://vt.tiktok.com/ZSataVEaX/',
 'id': '7582847487392632071',
 'title': '#perfume #fyp #poloblue #andrescroxatto #regalos  Polo Blue un perfum...',
 'duration_sec': 90,
 'uploader_name': 'capibara_store_oficial',
 'uploader_id': '7426875402553328646',
 'uploader_url': 'https://www.tiktok.com/@capibara_store_oficial',
 'has_music': True,
 'track': 'sonido original',
 'artist': 'Capibara Store',
 'view_count': 857500,
 'like_count': 29900,
 'comment_count': 318,
 'repost_count': 3696,
 'profile_unique_id_guess': 'capibara_store_oficial',
 'profile_name': 'Capibara Store',
 'followers': 13400,
 'following': 50,
 'profile_status': 'ok'}

###A) Mejorar la transcripción (rápido)
####1) Ver timestamps y segmentos (para validar)

In [ ]:
import json

with open("/content/tiktok/segments.json", "r", encoding="utf-8") as f:
    segs = json.load(f)

for s in segs[:10]:
    print(f"[{s['start']:.2f}–{s['end']:.2f}] {s['text']}")

[0.00–6.12]  perfume que ganó el campeonato de perfumes de Chet Noir cuando hizo el mundial de perfumes, el
[6.12–13.52]  ganador número uno por lo blue o de Parfum, qué cosa más espectacular, pero tiene que ser lo de Parfum,
[13.52–18.84]  no el Parfum, no el de lo de Toalé, tiene que ser este que es como el que trae todo el espíritu del
[18.84–24.36]  original, de lo de Toalé original, porque después el Parfum se fue, el aroma hizo otra cosa, pero
[24.44–30.48]  este trae todo el espíritu de la creación original, que es una cosa bien fresca, increíblemente
[30.48–36.92]  encantadora, diferente, super sexy, super super sexy, pero este es como concentrado y a parte tiene como
[36.92–42.36]  un golpe extra de exclusividad, porque un perfume mucho más denso, que tiene como una parte como
[42.36–49.68]  de algas, así como como de cuero, una cosa así, densa, rica, masculina, una cosa bien especial,
[49.72–54.32]  este es uno de los perfumes favoritos de mi esposa de toda la vida, cada vez 

⛳ Esto te permite ver si “Cuando falla la Matrix.” está bien capturado o se comió contexto.

### Para guardar los substitulos
Ahora se guarda la información en el laptop

In [ ]:
!whisper "/content/tiktok/audio.wav" --model small --language es --output_dir "/content/tiktok" --output_format srt
!ls -la /content/tiktok/*.srt

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
[00:00.000 --> 00:06.120]  perfume que ganó el campeonato de perfumes de Chet Noir cuando hizo el mundial de perfumes, el
[00:06.120 --> 00:13.520]  ganador número uno Polo Blue o de Parfum qué cosa más espectacular pero tiene que ser lo de Parfum
[00:13.520 --> 00:18.840]  no el Parfum no de lo de Toalé tiene que ser este que es como el que trae todo el espíritu del
[00:18.840 --> 00:24.360]  original de lo de Toalé original porque después el Parfum se fue el aroma hizo otra cosa pero
[00:24.400 --> 00:30.480]  este trae todo el espíritu de la creación original que una cosa bien fresca increíblemente
[00:30.480 --> 00:36.920]  encantadora diferente super sexy super super sexy. Este es como concentrado y a parte tiene como
[00:36.920 --> 00:42.360]  un golpe extra de exclusividad porque un pe

se descarga la info

In [ ]:
from google.colab import files
files.download("/content/tiktok/transcripcion.txt")
files.download("/content/tiktok/segments.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>